<a href="https://colab.research.google.com/github/shaikhisg/balkhash/blob/main/scripts/NDVI_Reclamation_Monitoring.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [9]:
import ee
import geemap
import calendar

# 1. АВТОРИЗАЦИЯ
PROJECT_ID = 'phd-monitoring-project'
try:
    ee.Initialize(project=PROJECT_ID)
except:
    ee.Authenticate()
    ee.Initialize(project=PROJECT_ID)

# --- ПАРАМЕТРЫ ---
LAT, LON = 53.788037, 88.162821
ROI = ee.Geometry.Point([LON, LAT]).buffer(10000).bounds()

ndvi_palette = ['#a50026', '#d73027', '#f46d43', '#ffffff', '#e6f598', '#abdb67', '#66bd63', '#1a9850']
ndvi_labels = ['-0.2', '-0.1', '0.0', '0.1', '0.2', '0.3', '0.4', '0.5']
ndvi_vis = {'min': -0.2, 'max': 0.5, 'palette': ndvi_palette}

Map = geemap.Map()
Map.setCenter(LON, LAT, 12)

# 2. ФУНКЦИЯ ПОЛУЧЕНИЯ NDVI (универсальная)
def get_ndvi_for_date(year, month):
    start = ee.Date.fromYMD(year, month, 1)
    end = start.advance(28, 'day')
    img = (ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED')
           .filterBounds(ROI)
           .filterDate(start, end)
           .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', 20))
           .median().clip(ROI))
    return img.normalizedDifference(['B8', 'B4'])

# 3. ЦИКЛ ДЛЯ ИСТОРИЧЕСКИХ СЛОЕВ (2022-2024)
YEARS = [2022, 2023, 2024]
MONTHS = [5, 7, 9]

for year in YEARS:
    for month in MONTHS:
        ndvi = get_ndvi_for_date(year, month)
        Map.addLayer(ndvi, ndvi_vis, f'NDVI {calendar.month_name[month]} {year}', False)

# --- ИСПРАВЛЕННЫЙ ПРОГНОЗ (Решаем проблему со скрина image_a61732.png) ---
def calculate_trend(year_num):
    # Используем функции ee.Number и ee.Date для работы на стороне сервера
    year = ee.Number(year_num)
    ndvi = get_ndvi_for_date(year, 7) # Берем июль для тренда
    # Добавляем временную полосу 't' (годы с 2022)
    return ndvi.addBands(ee.Image(year.subtract(2022)).float().rename('t'))

# Создаем коллекцию через ee.List для стабильности
years_ee = ee.List(YEARS)
collection = ee.ImageCollection(years_ee.map(calculate_trend))

# Считаем линейный тренд
trend = collection.reduce(ee.Reducer.linearFit())

# NDVI_2027 = (scale * 5 лет) + offset
ndvi_2027 = trend.select('scale').multiply(5).add(trend.select('offset')).clip(ROI)

Map.addLayer(ndvi_2027, ndvi_vis, 'PROJECTION 2027', True)

# 4. ЛЕГЕНДА
Map.add_legend(title="NDVI Vegetation Index", colors=ndvi_palette, labels=ndvi_labels)

Map

Map(center=[53.788037, 88.162821], controls=(WidgetControl(options=['position', 'transparent_bg'], widget=Sear…